# Hub-and-spoke: interactive walkthrough

Every spoke talks **only** to the hub; the hub owns the registry, the scenes and the routing.

The notebook must live next to `hub_and_spoke.py` (it imports it by name). Launch with:

```bash
uv run --with jupyterlab jupyter lab hub_and_spoke.ipynb
```

In [1]:
from hub_and_spoke import Hub, Spoke

## Build the hub and its spokes

This is the setup at the top of `main()`. `scenes` = desired state per scene, `routes` = event → (target spoke, command). Re-run this cell to reset all state.

In [3]:
hub = Hub(
    scenes={
        "movie":   {"light": "dim 20%", "blinds": "closed", "thermostat": "21°C"},
        "morning": {"light": "on",      "blinds": "open",   "thermostat": "23°C"},
    },
    routes={"motion": ("light", "on")},   # event → (target spoke, command)
)
hub

Hub(spokes={}, scenes={'movie': {'light': 'dim 20%', 'blinds': 'closed', 'thermostat': '21°C'}, 'morning': {'light': 'on', 'blinds': 'open', 'thermostat': '23°C'}}, routes={'motion': ('light', 'on')})

In [6]:
light, blinds, sensor = (Spoke(n, hub) for n in ("light", "blinds", "sensor"))
thermostat = Spoke("thermostat", hub)   # each Spoke registers itself with the hub

list(hub.spokes)

['light', 'blinds', 'sensor', 'thermostat']

## 1. Hub → spokes (fan-out)

One hub call becomes N spoke calls. The hub initiates, the spokes obey. Try `"morning"` too.

In [7]:
receipts = hub.apply_scene("movie")
receipts

['light: now dim 20%', 'blinds: now closed', 'thermostat: now 21°C']

## 2. Spoke → hub → spoke (routed)

The sensor sees motion and reports it. It has no idea the light exists; the hub's `routes` table turns `motion` into "light: on".

In [4]:
sensor.report("motion")
light.state  # the sensor never touched the light — the hub did

  hub: sensor said 'motion' → light: now on


'on'

## 3. Unknown event

The hub decides, not the spoke. An event with no route is ignored. Try adding a route with `hub.routes["wind gust"] = ("blinds", "closed")` and re-run.

In [5]:
blinds.report("wind gust")
blinds.state  # unchanged: nobody is routed to react

  hub: blinds said 'wind gust' — no route, ignored


'closed'

## 4. Wiring cost for N devices

Why the star wins as N grows: hub-and-spoke needs N links, a full mesh needs N(N−1)/2.

In [6]:
for n in (3, 5, 10, 50):
    print(f"N={n:>2}  hub-and-spoke: {n:>4} links   mesh: {n * (n - 1) // 2:>5} links")

N= 3  hub-and-spoke:    3 links   mesh:     3 links
N= 5  hub-and-spoke:    5 links   mesh:    10 links
N=10  hub-and-spoke:   10 links   mesh:    45 links
N=50  hub-and-spoke:   50 links   mesh:  1225 links


## 5. The price: no hub, no coordination

Single point of failure. A spoke without a hub cannot even construct itself.

In [7]:
try:
    Spoke("orphan", hub=None)  # type: ignore[arg-type]
except AttributeError as e:
    print("orphan spoke cannot even register — nothing to register with")
    print(f"AttributeError: {e}")

orphan spoke cannot even register — nothing to register with
AttributeError: 'NoneType' object has no attribute 'register'
